In [ ]:
import torch
import torch.nn as nn
import numpy as np
from scipy.special import logsumexp
from scipy.special import softmax


class AttentionModel(nn.Module):
    def __init__(
        self, 
        H, 
        d, 
        N, 
        q, 
        Q=None,
        V=None,
        K=None,
        lambd=0.001, 
        index_last_domain1=0, 
        H1=0, 
        H2=0, 
        init_fun=np.random.randn,
        device = 'cpu'
        
    ):
        super(AttentionModel, self).__init__()

        # Device & dtype
        # You could choose your own logic for picking device, or force CPU/GPU.
        # For example:
        #self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #self.device = torch.device("cpu")  
        self.dtype = torch.float32  # or torch.float64, etc.

        # Store hyperparameters
        self.H = H
        self.d = d
        self.N = N
        self.q = q
        self.lambd = lambd
        self.index_last_domain1 = index_last_domain1
        self.H1 = H1
        self.H2 = H2
        self.device = device
        self.Q=Q
        self.V=V
        self.K=K
        seed=0
        import random
        import numpy as np
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        # Define parameters
        if Q==None:
            self.Q = nn.Parameter(
                torch.tensor(np.random.randn(H, d, N), dtype=self.dtype, device=self.device)
            )
        if K==None:
            self.K = nn.Parameter(
                torch.tensor(np.random.randn(H, d, N), dtype=self.dtype, device=self.device)
            )
        if V==None:
            self.V = nn.Parameter(
                torch.tensor(np.random.randn(H, q, q), dtype=self.dtype, device=self.device)
            )
        import os
        cwd = os.getcwd()


        def read_tensor_from_txt(filename):
            with open(filename, 'r') as f:
                lines = f.readlines()

            # Read the dimensions from the first line
            dims = list(map(int, lines[0].strip().split()))

            # Initialize a list to hold the tensor data
            tensor_data = []

            current_slice = []
            for line in lines[1:]:
                line = line.strip()
                if line.startswith("Slice"):
                    if current_slice:  # If there is an existing slice, save it
                        tensor_data.append(current_slice)
                        current_slice = []
                elif line:  # Process non-empty lines
                    current_slice.append(list(map(float, line.split(','))))

            if current_slice:  # Append the last slice
                tensor_data.append(current_slice)

            # Convert the list back into a tensor with the original dimensions
            tensor = torch.tensor(tensor_data).view(*dims)
            return tensor
        

        # K = read_tensor_from_txt( cwd +"/results/34_23_save_random_without_J_400/K_tensor.txt")
        # Q= read_tensor_from_txt( cwd +"/results/34_23_save_random_without_J_400/Q_tensor.txt")
        # V = read_tensor_from_txt( cwd +"/results/34_23_save_random_without_J_400/V_tensor.txt")



        # self.Q.data = Q
        # self.K.data = K
        # self.V.data = V

    def forward(self, Z, weights):
        # Forward just calls the loss, as in your original code
        print("Inside forward method of AttentionModel")
        loss_value = self.loss_wo_J(
            self.Q, 
            self.K, 
            self.V, 
            Z, 
            weights,
            lambd=self.lambd,
            index_last_domain1=self.index_last_domain1,
            H1=self.H1,
            H2=self.H2
        )
        print("checking correspondance...")
        Q_np = self.Q.detach().cpu().numpy().copy()
        K_np = self.K.detach().cpu().numpy().copy()
        V_np = self.V.detach().cpu().numpy().copy()
        self.check_correspondance(Z, weights, Q_np, K_np, V_np, loss_value.item())
        return loss_value

    ###########################################################################
    #                            Helper Methods                                #
    ###########################################################################

    def create_attention_masks(self, H, L, index_last_domain1, H1, H2):

        device = self.device
        # Initialize masks tensor
        index_first_domain2 = index_last_domain1 + 1
        masks = torch.zeros(H, L, L, device=device)

        # Define head indices for each type
        H_total = H

        # Create position indices
        positions = torch.arange(L, device=device)

        # Create domain masks
        domain1_mask = (
            (positions.unsqueeze(1) <= index_last_domain1) 
            & (positions.unsqueeze(0) <= index_last_domain1)
        )
        domain2_mask = (
            (positions.unsqueeze(1) >= index_first_domain2) 
            & (positions.unsqueeze(0) >= index_first_domain2)
        )
        inter_domain_mask = (
            ((positions.unsqueeze(1) <= index_last_domain1) 
             & (positions.unsqueeze(0) >= index_first_domain2))
            | ((positions.unsqueeze(1) >= index_first_domain2) 
               & (positions.unsqueeze(0) <= index_last_domain1))
        )

        # Assign masks to heads
        for h in range(H_total):
            if h < H1:
                # Heads for Domain 1
                masks[h] = domain1_mask.float()
            elif h < H2:
                # Heads for Domain 2
                masks[h] = domain2_mask.float()
            else:
                # Heads for inter-domain interactions
                masks[h] = inter_domain_mask.float()

        return masks

    def compute_product_Q_K(self, Q, K):
        
        device = self.device
        dtype = self.dtype

        H, _, N = Q.shape
        # Step 1: Compute the raw attention scores using einsum
        e = torch.einsum('hdi,hdj->ijh', Q, K)  # Shape: (N, N, H)

        # Exclude self-interactions by setting scores to -inf on the diagonal
        #i_indices = torch.arange(N, device=device).unsqueeze(1)
        #j_indices = torch.arange(N, device=device).unsqueeze(0)
        #self_mask = (i_indices != j_indices).float()
        #mask_value = -1e9  # A large negative value to zero out after softmax
        #e = e * self_mask.unsqueeze(-1) + (1 - self_mask.unsqueeze(-1)) * mask_value

        # If there's a domain split:
        if self.index_last_domain1 > 0 and self.index_last_domain1 < N:
            domain_masks = self.create_attention_masks(
                H=H, 
                L=N, 
                index_last_domain1=self.index_last_domain1,
                H1=self.H1,  # per your original usage
                H2=self.H2
            )
            # Invert the domain masks to identify positions to mask
            inverted_domain_masks = (1 - domain_masks).bool()  # Positions to mask are True
            # Permute e to match the shape of domain_masks
            e = e.permute(2, 0, 1)  # Shape: (H, N, N)
            # Apply the masks
            e = e.masked_fill(inverted_domain_masks, mask_value)
            # Permute e back to original shape
            e = e.permute(1, 2, 0)  # Shape: (N, N, H)
        else:
            domain_masks = 0

        return e

    def compute_attention_heads(self, Q, K, V, index_last_domain1=0, H1=0, H2=0):
        
        device = self.device
        dtype = self.dtype

        H, _, N = Q.shape
        # N, _, _ = V.shape  # Actually your code re-assigns same N but let's keep it as is
        _N, _, _ = V.shape
        index_first_domain2 = index_last_domain1 + 1

        # Get e from compute_product_Q_K
        e = self.compute_product_Q_K(Q, K)

        sf = torch.zeros(N, N, H, device=device, dtype=dtype)
        for h in range(H):
            if index_last_domain1 != 0:
                if h < H1:
                    # Heads for Domain 1
                    softmax_vals = torch.softmax(
                        e[0:index_last_domain1+1, 0:index_last_domain1+1, h], 
                        dim=1
                    )
                    top = torch.cat([
                        softmax_vals,
                        torch.zeros(
                            index_last_domain1+1, 
                            N - index_last_domain1 - 1, 
                            device=device
                        )
                    ], dim=1)
                    sf_domain = torch.cat([
                        top,
                        torch.zeros(
                            N - index_last_domain1 - 1, 
                            N, 
                            device=device
                        )
                    ], dim=0)
                    sf = sf.clone()
                    sf[:, :, h] = sf_domain
                elif h < H2:
                    # Heads for Domain 2
                    softmax_vals = torch.softmax(
                        e[index_first_domain2:, index_first_domain2:, h], 
                        dim=1
                    )
                    # Create the top-left zero block
                    bottom_left = torch.zeros(
                        N - index_first_domain2, 
                        index_first_domain2, 
                        device=device
                    )
                    # top and bottom
                    top = torch.zeros(index_first_domain2, N, device=device)
                    bottom = torch.cat([bottom_left, softmax_vals], dim=1)
                    sf_domain = torch.cat([top, bottom], dim=0)
                    sf = sf.clone()
                    sf[:, :, h] = sf_domain
                else:
                    # Heads for inter-domain interactions
                    sf_domain = torch.softmax(e[:, :, h], dim=1)
                    sf = sf.clone()
                    sf[:, :, h] = sf_domain
            else:
                # No domain masks applied
                sf_domain = torch.softmax(e[:, :, h], dim=1)
                sf = sf.clone()
                sf[:, :, h] = sf_domain

        return sf

    def compute_mat_ene(self, Q, K, V, Z, H1=0, H2=0, index_last_domain1=0):
        
        # We'll assume you intended to call self.compute_attention_heads here.
        # The old snippet references 'e' out of nowhere, so presumably that was from compute_product_Q_K.
        # We'll keep the lines exactly the same, except we clarify how 'sf' is obtained.

        # The code below uses variables that appear in the snippet,
        # but to keep it consistent, let's define them in place:
        device = self.device
        dtype = self.dtype

        H, q, _ = V.shape
        # For clarity in your snippet, 'e' was from compute_product_Q_K,
        # and 'sf' is from compute_attention_heads.
        # We'll compute them inside to match your logic:

        sf = self.compute_attention_heads(
            Q=Q, 
            K=K, 
            V=V, 
            index_last_domain1=index_last_domain1, 
            H1=H1, 
            H2=H2
        )
        
        # From your snippet, you used e.shape for N_e1, N_e2, H_e,
        # but actually let's just read from sf itself.
        N_e1, N_e2, H_e = sf.shape
        N_Z, M = Z.shape

        assert N_e1 == N_e2 == N_Z, "Mismatch in N between sf and Z"
        N = N_e1

        i_indices = torch.arange(N).unsqueeze(1)
        j_indices = torch.arange(N).unsqueeze(0)
        self_mask = (i_indices != j_indices).float().to(Q.device)

        sf = sf * self_mask.unsqueeze(-1)

        mat_ene = torch.zeros(N, q, M, device=device, dtype=dtype)

        # Weighted sum loop
        for h in range(H):
            V_h = V[h]
            # The next line in your snippet references V_h[:, Z], 
            # but that can be tricky because Z is shape (N, M).
            # We keep it as it is in your snippet, trusting you have reason:
            V_h_Zj = V_h[:, Z]     # shape => (q, N, M)
            V_h_Zj = V_h_Zj.permute(1, 0, 2)  # => (N, q, M)

            mat_ene_h = torch.einsum('ij,jqm->iqm', sf[:, :, h], V_h_Zj)
            mat_ene += mat_ene_h

        mat_ene = mat_ene.permute(1, 0, 2)
        return mat_ene, sf

    def loss_wo_J(self, Q, K, V, Z, weights, lambd=0.001, index_last_domain1=0, H1=0, H2=0):
        
        device = self.device
        dtype = self.dtype

        H, d, N = Q.shape
        q = V.shape[1]  # Number of amino acids
        M = Z.shape[1]

        # Step: compute mat_ene and sf
        mat_ene, sf = self.compute_mat_ene(
            Q, 
            K, 
            V, 
            Z, 
            H1=H1, 
            H2=H2, 
            index_last_domain1=index_last_domain1
        )  # Shape: (q, N, M)
        # logsumexp
        lge = torch.logsumexp(mat_ene, dim=0)  # Shape: (N, M)

        Z_indices = Z.unsqueeze(0)  # Shape: (1, N, M)
        mat_ene_selected = torch.gather(mat_ene, dim=0, index=Z_indices).squeeze(0)  # (N, M)

        pl_elements = weights * (mat_ene_selected - lge) #weighted sum along M
        pl = -torch.sum(pl_elements) # sum along N
        print('pl (ale):', pl.item())
        # For the regularization term, your snippet references M_matrix, etc.
        # That part of your snippet uses `self_mask`, but it was never fully spelled out. 
        # We'll keep it exactly as in your snippet:

        # The snippet tries: M_matrix = torch.einsum('ijh,ijk,ij->hk', sf, sf, self_mask)
        # But 'self_mask' is not defined in this scope. If that was part of your code, 
        # you must define it. We'll keep the line as is (though it may error if `self_mask` is missing).
    # Compute regularization term
        i_indices = torch.arange(N, device=device).unsqueeze(1)
        j_indices = torch.arange(N, device=device).unsqueeze(0)
        self_mask = (i_indices != j_indices).float()
        M_matrix = torch.einsum('ijh,ijk,ij->hk', sf, sf, self_mask)  # Shape: (H, H)
        VV = V.view(H, -1)  # Shape: (H, q*q)
        VV_T = VV @ VV.T  # Shape: (H, H)
        sum_J_squared = torch.sum(M_matrix * VV_T)  # Scalar
        print('sum_J_squared:', sum_J_squared.item())
        Q_np = self.Q.detach().cpu().numpy().copy()
        K_np = self.K.detach().cpu().numpy().copy()
        V_np = self.V.detach().cpu().numpy().copy()
        J = self.compute_J(Q_np, K_np, V_np)
        sum_J_squared_hand = np.sum(J**2)
        print('sum_J_squared by hand:', np.sum(J**2))

        reg = lambd * sum_J_squared  # Scalar
        #reg = lambd * sum_J_squared_hand
        loss_value = pl + reg
        print('reg (ale):', reg.item())
        print('pl (ale):', pl.item())
        del sf, mat_ene, mat_ene_selected, M, VV_T
        torch.cuda.empty_cache()

        return loss_value
    ######

    def softmax_function(self,Q,K):
        """
        Compute the softmax of the dot product of Q and K^T - for one head.
        """
        H = Q.shape[0]  # Number of heads
        d = Q.shape[1]  # Dimension of each head
        L = Q.shape[2]  # Length of the sequence
        sf = np.zeros((H, L, L))  # Initialize softmax scores
        product = np.einsum('hdi,hdj->hij', Q, K)  # Compute dot product for all heads at once
        e = self.compute_product_Q_K(self.Q, self.K)
        e = e.detach().cpu().numpy()
        e = e.transpose(2, 0, 1)  # shape (H, L, L)
        print("product shape:", e.shape)
        # check if equal to product
        assert np.allclose(e, product, atol=1e-5), "Dot products do not match!"
        print("Difference in dot products:", np.max(np.abs(e - product)))   
        sf = softmax(e, axis=-1) # shape (H, L, L)
        return sf
    
    def value_product(self,sf,V):
        """
        Compute the product of the softmax scores sf and the value matrix V,
        with the mask j != I applied.

        sf: shape (L, L, H)
        V:  shape (H,q, q)

        Returns:
            J: shape (L, L, q, q)
        """
        print("sf shape:", sf.shape)
        # Contract over the H dimension
        J = np.einsum('hij,hab->ijab', sf, V)

        # Apply the mask j != I
        L = sf.shape[1]
        mask = np.ones((L, L))
        np.fill_diagonal(mask, 0)
        print("mask shape:", mask.shape)
        print("shape of broadcasted mask:", mask[None, None, :, :].shape)
        J *= mask[:,:,None, None]  # broadcast mask over q, q dimensions
        return J

    def compute_J(self, Q, K, V):
        """
        Compute the attention matrix J using the formula:
        J = softmax(Q @ K^T) @ V
        Q, K have shape (H,d,L)
        """
        sf = self.softmax_function(Q,K)
        J = self.value_product(sf, V)
        sf_ale = self.compute_attention_heads(self.Q, self.K, self.V)
        sf_ale = sf_ale.detach().cpu().numpy()
        sf_ale = np.transpose(sf_ale, (2, 0, 1))  # shape (H, L, L)
        J_ale = self.value_product(sf_ale, V)
        # check if J and J_ale are equal within tolerance, if not raise error
        assert np.allclose(J, J_ale, atol=1e-5), "J from hand and model do not match!"
        print("Difference in J:", np.max(np.abs(J - J_ale)))
        J_sum = np.sum(J**2)
        J_ale_sum = np.sum(J_ale**2)
        print("Sum of squares of J (ale):", J_ale_sum)
        print("Sum of squares of J (hand):", J_sum)
        return J
    
    def compute_mat_ene_hand(self, J, Z):
        """
        Compute mat_ene[a, r, m] = sum_j J[r, j, a, Z[j,m]]
        J: (L, L, A, B)
        Z: (M,L) integer array of sequences
        """
        L, _, A, B = J.shape
        Z = np.asarray(Z, dtype=int)
        if Z.ndim == 1:
            Z = Z.reshape(L, 1)
        elif Z.shape[0] != L and Z.shape[1] == L:
            Z = Z.T
        M = Z.shape[1]

        mat_ene = np.zeros((A, L, M))
        for a in range(A):
            for r in range(L):
                for m in range(M):
                    s = 0.0
                    for j in range(L):
                        s += J[r, j, a, Z[j, m]]
                    mat_ene[a, r, m] = s
        return mat_ene

    def compute_lge(self, mat_ene):
        """
        Julia equivalent of lge = logsumexp(mat_ene, dims=1)[1,:,:]
        mat_ene: (A, L, M)
        """
        # logsumexp along axis=0 (the 'a' dimension)
        lge = logsumexp(mat_ene, axis=0)  # shape (L, M)
        return lge
    
    def compute_pl(self, mat_ene, lge, Z, weights):
        """
        Equivalent of Julia:
        @tullio pl = weights[m] * (mat_ene[Z[r,m], r, m] - lge[r,m])
        
        mat_ene: (A, L, M)
        lge:     (L, M)
        Z:       (M,L) with integer indices (0-based!)
        weights: (M,)
        """
        # Z to array
        Z=Z.T
        M,L = Z.shape  # sequence length × number of sequences
        print("M:", M)
        print("L:", L)
        pl = 0.0
        for m in range(M):
            for r in range(L):
                a = int(Z[m,r])   # pick correct alphabet index
                pl += (mat_ene[a, r, m] - lge[r, m])*weights[m]
        pl = -1*pl
        return pl
    
    def loss_by_hand(self,J, Z, weights, lambd=0.001):
        Z = np.asarray(Z, dtype=int)
        weights = np.asarray(weights, dtype=float)
        mat_ene = self.compute_mat_ene_hand(J, Z)
        lge = self.compute_lge(mat_ene)
        print("mat_ene shape:", mat_ene.shape)
        print("lge shape:", lge.shape)
        print("Z shape:", Z.shape)
        print("weights shape:", weights.shape)
        pl = self.compute_pl(mat_ene, lge, Z, weights)
        reg = lambd * np.sum(J ** 2)
        print("pl:", pl)
        print("reg:", reg)
        return pl + reg

    # functions to check correspondance
    def check_correspondance(self, Z, weights, Q, K, V, loss):
        J = self.compute_J(Q, K, V)
        # check difference in softmax
        sf_hand = self.softmax_function(Q, K)
        sf = self.compute_attention_heads(self.Q, self.K, self.V)
        sf = sf.detach().cpu().numpy()
        sf = np.transpose(sf, (2, 0, 1))  # shape (H, L, L)
        assert np.allclose(sf_hand, sf, atol=1e-5), "Softmax outputs do not match!"
        print("Difference in softmax:", np.max(np.abs(sf_hand - sf)))
        mat_ene_hand = self.compute_mat_ene_hand(J, Z)
        mat_ene = self.compute_mat_ene(self.Q,self.K,self.V,Z)
        mat_ene = mat_ene[0].detach().cpu().numpy()
        # check if two mat_ene equal within tolerance, if not raise error
        assert np.allclose(mat_ene_hand, mat_ene, atol=1e-5), "mat_ene from hand and model do not match!"
        print("Difference in mat_ene:", np.max(np.abs(mat_ene_hand - mat_ene)))
        #diff = np.max(np.abs(mat_ene_hand - mat_ene.detach().cpu().numpy()))
        #print("Max difference in mat_ene:", diff)
        loss_hand = self.loss_by_hand(J, Z, weights)
        print("Loss by hand:", loss_hand)
        print("Loss by model:", loss)
        print("Difference in loss:", np.abs(loss_hand - loss))
        return 0

In [15]:
model = AttentionModel(H=64, d=10, N=63, q=21, lambd=0.001, index_last_domain1=0, H1=0, H2=0, device='cpu')

# Z random array of 63 integers between 0 and 20, shape (1,63)
row1 = [11, 12, 19, 3, 17, 9, 5, 7, 8, 3, 5, 0, 15, 13, 2, 3, 7, 8, 8, 0,
     19, 14, 3, 9, 0, 8, 8, 19, 6, 12, 2, 13, 19, 5, 2, 11, 12, 9, 0, 3,
     3, 8, 10, 14, 3, 9, 11, 3, 0, 19, 2, 19, 9, 15, 8, 11, 15, 20, 1, 4, 6, 7, 2]  # 63 elements
row2 = [5, 7, 8, 3, 5, 0, 15, 13, 2, 3, 7, 8, 8, 0, 19, 14, 3, 9, 11, 12,
     19, 3, 17, 9, 5, 7, 8, 3, 5, 0, 15, 13, 2, 3, 7, 8, 8, 0, 19, 14,
     3, 9, 0, 8, 8, 19, 6, 12, 2, 13, 19, 5, 1, 2, 4, 6, 7, 0, 1, 2,4,18,7]   # 63 elements
Z = np.array([row1, row2])
print(Z.shape) 
Z = torch.tensor(Z, dtype=torch.long)
weights = (torch.ones(Z.shape[0], dtype=torch.float32) / Z.shape[0])
model.forward(Z.T, weights)

(2, 63)
Inside forward method of AttentionModel
pl (ale): 633.1553955078125
sum_J_squared: 577997.5625
product shape: (64, 63, 63)
Difference in dot products: 3.8146973e-06
sf shape: (64, 63, 63)
mask shape: (63, 63)
shape of broadcasted mask: (1, 1, 63, 63)
sf shape: (64, 63, 63)
mask shape: (63, 63)
shape of broadcasted mask: (1, 1, 63, 63)
Difference in J: 1.1920929e-06
Sum of squares of J (ale): 577999.44
Sum of squares of J (hand): 577999.4
sum_J_squared by hand: 577999.4
reg (ale): 577.9976196289062
checking correspondance...
product shape: (64, 63, 63)
Difference in dot products: 3.8146973e-06
sf shape: (64, 63, 63)
mask shape: (63, 63)
shape of broadcasted mask: (1, 1, 63, 63)
sf shape: (64, 63, 63)
mask shape: (63, 63)
shape of broadcasted mask: (1, 1, 63, 63)
Difference in J: 1.1920929e-06
Sum of squares of J (ale): 577999.44
Sum of squares of J (hand): 577999.4
product shape: (64, 63, 63)
Difference in dot products: 3.8146973e-06
Difference in softmax: 3.5762787e-07
Differen

tensor(1211.1531, grad_fn=<AddBackward0>)